# ARC/ATLAS Train Prep v4
Prep ARC + ATLAS raw T1/masks into standardized MNI data inside this folder, then combine into a training set (t1/masks). Re-run safely; it skips already processed roots unless overwrite=True.

In [1]:
from pathlib import Path
import csv

from src.data_prep.prep_utils import DatasetConfig, run_prep, combine_standardized

PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()

# Raw roots (edit if your data lives elsewhere)
ARC_RAW   = Path('/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/t1w_with_masks_raw')
ATLAS_IMG = Path('/home/rbielski/Atlas_2/Training/Images')
ATLAS_MSK = Path('/home/rbielski/Atlas_2/Training/Masks')

DATASETS = [
    DatasetConfig(name="ARC",   images_dir=ARC_RAW,  masks_dir=ARC_RAW),
    DatasetConfig(name="ATLAS", images_dir=ATLAS_IMG, masks_dir=ATLAS_MSK, t1_glob='**/*.nii.gz', mask_glob='**/*.nii.gz'),
]

OUT_ROOT = PROJECT_ROOT / "data" / "prep_outputs"
outputs = run_prep(DATASETS, OUT_ROOT)
print("processed roots:", outputs)

COMBINED = PROJECT_ROOT / "data" / "processed" / "train_combined"
combine_standardized(outputs, COMBINED)


def enforce_arc_naming(combined_root: Path) -> None:
    """Keep only ARC-t1w-standardized-* ARC entries and rebuild manifest."""
    t1_dir = combined_root / "t1"
    mask_dir = combined_root / "masks"

    removed = 0
    renamed = 0

    def _rewrite_dir(folder: Path) -> None:
        nonlocal removed, renamed
        for p in sorted(folder.glob('*.nii.gz')):
            if '__' not in p.name:
                continue
            slug, rest = p.name.split('__', 1)
            if slug.startswith('ARC-standardized-'):
                p.unlink()
                removed += 1
                continue
            if slug.startswith('ARC-t1w-with-masks-raw-'):
                new_slug = slug.replace('ARC-t1w-with-masks-raw-', 'ARC-t1w-standardized-', 1)
                q = folder / f"{new_slug}__{rest}"
                if q.exists():
                    # Keep a single canonical copy if this notebook has already been run.
                    p.unlink()
                else:
                    p.rename(q)
                renamed += 1

    _rewrite_dir(t1_dir)
    _rewrite_dir(mask_dir)

    rows = []
    for t1 in sorted(t1_dir.glob('*.nii.gz')):
        mask_name = t1.name.replace('_T1w_MNI_norm', '_lesion_mask_MNI_clean')
        mask = mask_dir / mask_name
        if not mask.exists():
            continue
        if '__' in t1.name:
            slug, key = t1.name.split('__', 1)
        else:
            slug, key = '', t1.name
        rows.append(
            {
                'slug': slug,
                'key': key,
                't1': str(t1.resolve()),
                'mask': str(mask.resolve()),
            }
        )

    mf = combined_root / 'manifest.csv'
    with mf.open('w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['slug', 'key', 't1', 'mask'])
        w.writeheader()
        w.writerows(rows)

    print(f'ARC naming enforced: removed={removed}, renamed={renamed}, manifest_rows={len(rows)}')


enforce_arc_naming(COMBINED)
print("combined set at", COMBINED)



[ARC] image root: /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/t1w_with_masks_raw | mask root: /home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/t1w_with_masks_raw
[ARC] globs: t1=**/*_T1w.nii.gz masks=**/*mask*.nii.gz
[ARC] images: 203 masks: 203 pairs found: 203
[ARC] already processed -> /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/prep_outputs/ARC-t1w-with-masks-raw-5893ef9b, skipping (overwrite=True to redo).
[ATLAS] image root: /home/rbielski/Atlas_2/Training/Images | mask root: /home/rbielski/Atlas_2/Training/Masks
[ATLAS] globs: t1=**/*.nii.gz masks=**/*.nii.gz
[ATLAS] images: 655 masks: 655 pairs found: 655
[ATLAS] already processed -> /home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4/data/prep_outputs/ATLAS-Images-f0d7431e, skipping (overwrite=True to redo).
No QC rows written (no datasets processed).
processed roots: [PosixPath('/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_

## Visualize standardized outputs
Use the viewer to spot-check T1/mask overlays in `train_combined` after prep and ARC name normalization (`ARC-t1w-standardized-*`).



In [1]:

from src.data_prep.viewer import show_viewer
show_viewer(PROJECT_ROOT / 'data' / 'processed' / 'train_combined')


NameError: name 'PROJECT_ROOT' is not defined

## ARC Naming Policy
This notebook keeps ARC entries in `train_combined` under `ARC-t1w-standardized-*` and removes legacy `ARC-standardized-*` entries.

